# Kidsignal 실제 BigQuery cohort intake

이 노트북은 전용 model-reader ADC로 authorized TRAIN/VALIDATION view 한 개만 읽고, 24-field 표준 plane 또는 26-field 행동 plane의 schema·UUIDv4·public digest·split을 검증합니다. 행 원본과 개인 식별자는 저장하지 않으며, 준비된 배열의 shape만 기록합니다.

성공 상태도 `READY_FOR_SYNC_NOT_TRAINED`이며 fit·평가·ONNX·bundle·승격을 실행하지 않습니다.

In [ ]:
import os
from pathlib import Path

from multisensor_ml.bigquery_training_preflight import GCP_LOCATION, GCP_PROJECT
from multisensor_ml.kaggle_bigquery_intake import (
    run_kaggle_bigquery_intake,
    write_kaggle_intake_artifact,
)

def required_env(name: str) -> str:
    value = os.environ.get(name, '').strip()
    if not value:
        raise RuntimeError(f'{name} must be configured externally')
    return value

MODE = required_env('KIDSIGNAL_INTAKE_MODE')
if MODE not in {'standard', 'behavior'}:
    raise RuntimeError('KIDSIGNAL_INTAKE_MODE must be standard or behavior')
COHORT_UUID = required_env('KIDSIGNAL_COHORT_UUID')
EXPECTED_PUBLIC_COHORT_DIGEST = required_env('KIDSIGNAL_PUBLIC_COHORT_DIGEST')
EXPECTED_PUBLIC_SPLIT_DIGEST = required_env('KIDSIGNAL_PUBLIC_SPLIT_DIGEST')
OUTPUT_PATH = Path(os.environ.get('KIDSIGNAL_INTAKE_OUTPUT', '/kaggle/working/kidsignal_bigquery_intake_receipt.json'))
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_ONNX_EXPORT = False
RUN_BUNDLE = False
RUN_PROMOTION = False
assert not any((RUN_TRAINING, RUN_EVALUATION, RUN_ONNX_EXPORT, RUN_BUNDLE, RUN_PROMOTION))

In [ ]:
from datetime import datetime, timezone

try:
    from google.cloud import bigquery
except ModuleNotFoundError as error:
    raise RuntimeError('google-cloud-bigquery is a late runtime dependency for this execution cell') from error

try:
    bigquery_client = bigquery.Client(project=GCP_PROJECT, location=GCP_LOCATION)
except Exception as error:
    raise RuntimeError('BigQuery ADC credentials are required at runtime') from error
credentials = getattr(bigquery_client, '_credentials', None)
observed_principal = getattr(credentials, 'service_account_email', None)
if not isinstance(observed_principal, str) or not observed_principal:
    raise RuntimeError('ADC model-reader principal could not be verified')

observed_at_utc = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
intake_artifact = run_kaggle_bigquery_intake(
    mode=MODE,
    client=bigquery_client,
    cohort_uuid=COHORT_UUID,
    expected_public_cohort_digest=EXPECTED_PUBLIC_COHORT_DIGEST,
    expected_public_split_digest=EXPECTED_PUBLIC_SPLIT_DIGEST,
    observed_at_utc=observed_at_utc,
    observed_principal=observed_principal,
)
write_kaggle_intake_artifact(OUTPUT_PATH, intake_artifact)

In [ ]:
import json

receipt = intake_artifact['readiness_receipt']
if receipt['fit_call_count'] != 0:
    raise RuntimeError('intake boundary attempted model fitting')
public_summary = {
    'mode': intake_artifact['mode'],
    'status': receipt['status'],
    'row_count': receipt['row_count'],
    'training_ready': receipt['training_ready'],
    'fit_call_count': receipt['fit_call_count'],
    'shape_summary': intake_artifact['summary'],
}
print(json.dumps(public_summary, ensure_ascii=False, sort_keys=True))

## 실행 경계

`BLOCKED_*`는 입력 계약 또는 실제 cohort 준비가 충족되지 않았음을 뜻합니다. `READY_FOR_SYNC_NOT_TRAINED`도 학습 승인이 아니며, 별도의 명시적 실제 cohort 학습 작업으로만 이어질 수 있습니다.